### Step 1 — Environment

Run **once per new kernel** (SageMaker notebook instance, local venv, etc.).

- Prefer installing from **`requirements-mlops.txt`** at the repo root so Git + SageMaker jobs share the same dependency list.
- Do **not** `pip install boto3` here (SageMaker: can break the pinned `awscli` / `botocore` stack).


In [ ]:
import subprocess
import sys
from pathlib import Path


def _find_repo_root() -> Path:
    """Walk upward from cwd until we see repo markers."""
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "requirements-mlops.txt").exists():
            return p
        if (p / "configs" / "train.yaml").exists():
            return p
    return cwd


_REPO = _find_repo_root()
_REQ = _REPO / "requirements-mlops.txt"

if _REQ.is_file():
    print(f"Installing from {_REQ} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(_REQ)])
else:
    print("requirements-mlops.txt not found; using inline pins (same as reference notebook).")
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "torch>=2.2.0",
            "pandas>=2.0.0",
            "numpy>=1.26.0",
            "scikit-learn>=1.4.0",
            "mlflow>=2.10.0",
            "pyarrow>=15.0.0",
            "s3fs>=2024.1.0",
            "polars>=0.20.0",
            "PyYAML>=6.0.0",
        ]
    )

print("Environment setup done.")


## Two-Tower end-to-end orchestration (thin notebook)

This notebook should stay **thin**: load config + call library functions.

All real logic lives in the `two_tower` package under `src/two_tower/`.


In [ ]:
# Dev mode: import `two_tower` from this repo without `pip install -e .`
import sys
from pathlib import Path


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "requirements-mlops.txt").exists():
            return p
        if (p / "configs" / "train.yaml").exists():
            return p
    raise FileNotFoundError(
        "Could not find repo root (looked for requirements-mlops.txt or configs/train.yaml). "
        "Open or `cd` to the Two_Tower repository root and restart the kernel."
    )


repo_root = find_repo_root()
sys.path.insert(0, str(repo_root / "src"))

print("Repo root:", repo_root)
print("Python path[0]:", sys.path[0])


### Step 2 — Load pipeline config (YAML → dataclasses)

Edit paths and feature lists in `configs/train.yaml`, then run the next cell.


In [ ]:
from two_tower.config_loader import load_pipeline_config

TRAIN_CONFIG = (repo_root / "configs" / "train.yaml").resolve()
pipeline_cfg = load_pipeline_config(TRAIN_CONFIG)
print("train path:", pipeline_cfg.paths.train)
print("label_col:", pipeline_cfg.features.label_col)


### Step 3 — Load train / validation Parquet

Uses `pd.read_parquet` on the paths in config (S3 or local). The `data:` section in `train.yaml` can:

- **`merge_client_metadata`**: left-join `client_*` from `client_metadata_uri` on `features.client_id_col` (one row per bundle, e.g. `com.grofers.customerapp`).
- **`inject_single_client_metadata`**: broadcast a single metadata row to every row (mutually exclusive with merge).


In [ ]:
from two_tower.data.load import load_train_validation_frames

train_df, val_df = load_train_validation_frames(pipeline_cfg)


### Step 3a — Optional per-client downsampling (train)

When multiple clients share one training table, a large client can dominate gradients. This step can:

1. **Balance neg:pos** within each `client_id_col` (target ratio `TRAIN_NEG_PER_POS : 1`).
2. **Equalize clients** so each client keeps the same number of rows (the minimum feasible across clients), preserving that ratio.

Set `DOWNSAMPLE_TRAIN = True` to enable. **Validation** is unchanged by default (`DOWNSAMPLE_VAL = False`).

In [ ]:
import pandas as pd

# --- toggles ---
DOWNSAMPLE_TRAIN = False
# Target negatives per positive within each client (None skips the ratio step entirely).
TRAIN_NEG_PER_POS = 10
# After ratio balancing, cap every client to the same row count (min across clients).
TRAIN_EQUALIZE_CLIENT_ROWS = True
DOWNSAMPLE_VAL = False
DOWNSAMPLE_RANDOM_STATE = 42


def _balance_train_per_client(
    df: pd.DataFrame,
    neg_per_pos: int | None,
    ycol: str,
    ccol: str,
    rs: int = 42,
) -> pd.DataFrame:
    """Undersample within each client so negatives:positives ≈ r:1 (no duplication)."""
    if neg_per_pos is None:
        return df
    r = max(1, int(neg_per_pos))
    parts = []
    for i, (_cid, g) in enumerate(df.groupby(ccol, sort=False)):
        pos = g[g[ycol] == 1]
        neg = g[g[ycol] == 0]
        n_p, n_n = len(pos), len(neg)
        if n_p == 0 or n_n == 0:
            parts.append(g)
            continue
        k = min(n_p, n_n // r)
        rs_p, rs_n = rs + 2 * i, rs + 2 * i + 1
        if k == 0:
            m = min(n_p, n_n)
            parts.append(
                pd.concat(
                    [pos.sample(n=m, random_state=rs_p), neg.sample(n=m, random_state=rs_n)],
                    ignore_index=True,
                )
            )
        else:
            parts.append(
                pd.concat(
                    [pos.sample(n=k, random_state=rs_p), neg.sample(n=k * r, random_state=rs_n)],
                    ignore_index=True,
                )
            )
    return pd.concat(parts, ignore_index=True) if parts else df.iloc[:0]


def _equalize_clients_to_min_rows_keep_ratio(
    df: pd.DataFrame,
    neg_per_pos: int | None,
    ycol: str,
    ccol: str,
    rs: int = 123,
) -> pd.DataFrame:
    """Downsample each client to the same row count while keeping neg:pos = r:1 per client."""
    if neg_per_pos is None:
        return df
    r = max(1, int(neg_per_pos))
    stats = []
    for _cid, g in df.groupby(ccol, sort=False):
        pos_c = int((g[ycol] == 1).sum())
        neg_c = int((g[ycol] == 0).sum())
        stats.append((_cid, pos_c, neg_c))
    if not stats:
        return df
    m = min(min(pos_c, (neg_c // r) if r else pos_c) for _cid, pos_c, neg_c in stats)
    if m <= 0:
        return df
    parts = []
    for i, (_cid, g) in enumerate(df.groupby(ccol, sort=False)):
        pos = g[g[ycol] == 1]
        neg = g[g[ycol] == 0]
        rs_p, rs_n = rs + 3 * i, rs + 3 * i + 1
        parts.append(
            pd.concat(
                [pos.sample(n=m, random_state=rs_p), neg.sample(n=m * r, random_state=rs_n)],
                ignore_index=True,
            )
        )
    return pd.concat(parts, ignore_index=True) if parts else df.iloc[:0]


def _print_client_split(name: str, df: pd.DataFrame, ycol: str, ccol: str) -> None:
    if len(df) == 0:
        print(f"{name}: empty")
        return
    per = (
        df.groupby(ccol, sort=False)[ycol]
        .agg(pos=lambda s: int((s == 1).sum()), neg=lambda s: int((s == 0).sum()), total="count")
        .reset_index()
    )
    print(f"{name} rows={len(df):,} pos={(df[ycol] == 1).sum():,} neg={(df[ycol] == 0).sum():,}")
    print(per.to_string(index=False))


ycol = pipeline_cfg.features.label_col
ccol = pipeline_cfg.features.client_id_col

if DOWNSAMPLE_TRAIN:
    _print_client_split("train (before)", train_df, ycol, ccol)
    train_before = len(train_df)
    train_work = _balance_train_per_client(
        train_df, TRAIN_NEG_PER_POS, ycol, ccol, rs=DOWNSAMPLE_RANDOM_STATE
    )
    if TRAIN_NEG_PER_POS is not None and TRAIN_EQUALIZE_CLIENT_ROWS:
        train_work = _equalize_clients_to_min_rows_keep_ratio(
            train_work, TRAIN_NEG_PER_POS, ycol, ccol, rs=DOWNSAMPLE_RANDOM_STATE + 1000
        )
    train_df = train_work.sample(frac=1.0, random_state=DOWNSAMPLE_RANDOM_STATE).reset_index(drop=True)
    print(
        f"\ntrain downsampling: {train_before:,} -> {len(train_df):,} "
        f"(neg:pos={TRAIN_NEG_PER_POS}:1, equalize={TRAIN_EQUALIZE_CLIENT_ROWS})\n"
    )
    _print_client_split("train (after)", train_df, ycol, ccol)
else:
    print("DOWNSAMPLE_TRAIN=False — skipped (set True to balance/equalize per client).")

if DOWNSAMPLE_VAL:
    _print_client_split("val (before)", val_df, ycol, ccol)
    val_before = len(val_df)
    val_work = _balance_train_per_client(
        val_df, TRAIN_NEG_PER_POS, ycol, ccol, rs=DOWNSAMPLE_RANDOM_STATE + 7
    )
    if TRAIN_NEG_PER_POS is not None and TRAIN_EQUALIZE_CLIENT_ROWS:
        val_work = _equalize_clients_to_min_rows_keep_ratio(
            val_work, TRAIN_NEG_PER_POS, ycol, ccol, rs=DOWNSAMPLE_RANDOM_STATE + 2000
        )
    val_df = val_work.sample(frac=1.0, random_state=DOWNSAMPLE_RANDOM_STATE + 7).reset_index(drop=True)
    print(
        f"\nval downsampling: {val_before:,} -> {len(val_df):,} "
        f"(neg:pos={TRAIN_NEG_PER_POS}:1, equalize={TRAIN_EQUALIZE_CLIENT_ROWS})\n"
    )
    _print_client_split("val (after)", val_df, ycol, ccol)

### Step 3b — Feature columns, vocabs, hash embedding tables

Intersects configured columns with Parquet, splits cat/num/multi, fits vocabs on **train** only (after Step 3a if you downsampled), builds hash init weights (reference notebook). Re-run after changing `features:` or `train.min_count` in YAML.


In [ ]:
from two_tower.features.prepare import prepare_training_features

feature_artifacts = prepare_training_features(train_df, val_df, pipeline_cfg)


### Step 4 — Train + MLflow + artifacts

Runs the full loop (BCE logits, val AUC, best-epoch checkpoint), logs to MLflow, and writes **user tower** `.pt`, **vocab** `.pkl`, and **client embeddings** Parquet under `paths.artifacts_base` (S3 or local). Requires `MLflow` and a valid `mlflow_tracking_uri` / local tracking if you use the server.


In [ ]:
import os
import subprocess
import sys

# Toggle: use multi-GPU DDP training (recommended for 4x GPU SageMaker).
# Note: DDP runs in separate processes, so it cannot reuse in-notebook `train_df/val_df/feature_artifacts`.
# It will load data from `configs/train.yaml` like the CLI script does.
USE_DDP = True

# Optional: enable per-epoch train metric sampling (helps spot overfitting: train >> val).
# Set to 0 to disable (default in library code).
TRAIN_EVAL_MAX_EXAMPLES = 200_000


if USE_DDP and torch.cuda.is_available() and torch.cuda.device_count() > 1:
    cfg_path = str(TRAIN_CONFIG)
    cmd = [
        "torchrun",
        f"--nproc_per_node={torch.cuda.device_count()}",
        str(repo_root / "scripts" / "train.py"),
        "--config",
        cfg_path,
    ]
    print("Launching DDP training:")
    print(" ".join(cmd))

    env = dict(os.environ)
    env["PYTHONPATH"] = str(repo_root / "src")
    env["OMP_NUM_THREADS"] = env.get("OMP_NUM_THREADS", "1")

    subprocess.check_call(cmd, env=env, cwd=str(repo_root))
else:
    # Single-process mode (uses the in-notebook dataframes + precomputed feature artifacts).
    from two_tower.training import train_and_log

    try:
        pipeline_cfg.train.train_eval_max_examples = TRAIN_EVAL_MAX_EXAMPLES
    except Exception:
        pass

    train_and_log(
        cfg=pipeline_cfg,
        train_df=train_df,
        val_df=val_df,
        feature_artifacts=feature_artifacts,
    )


### Step 5 — Batch inference

Set `paths.infer`, `paths.artifacts_base` (same as training), and `infer.ranking_output` in `configs/infer.yaml`. Artifacts must exist: `artifacts/user_tower/user_tower_state.pt`, `artifacts/vocab_artifact/vocab_artifact.pkl`, `artifacts/client_embeddings/client_embeddings.parquet`.


In [ ]:
from two_tower.config_loader import load_infer_job_config
from two_tower.inference.run import run_inference_job

INFER_CONFIG = (repo_root / "configs" / "infer.yaml").resolve()
infer_cfg = load_infer_job_config(INFER_CONFIG)
run_inference_job(infer_cfg)
